# RetailRocket - Dummy and Baseline Experiments

Objetivo: criar uma primeira esteira de recomendacao com split temporal, dummy de popularidade e baseline colaborativo item-item, com metricas de ranking.

## Etapa 1 - Imports e configuracoes

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import mlflow

RANDOM_STATE = 42
TOP_K = 10
EVENT_WEIGHT = {"view": 1.0, "addtocart": 3.0, "transaction": 5.0}

## MLflow Setup - Rastreamento de Experimentos

Inicializamos MLflow para registrar cada variação de baseline com seus parâmetros e métricas.

In [3]:
# Configurar backend local do MLflow
MLFLOW_TRACKING_URI = Path("../mlruns").absolute()
mlflow.set_tracking_uri(f"file:{MLFLOW_TRACKING_URI}")
mlflow.set_experiment("baseline_experiments")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('baseline_experiments').name}")

2026/06/23 20:50:48 INFO mlflow.tracking.fluent: Experiment with name 'baseline_experiments' does not exist. Creating a new experiment.


MLflow tracking URI: file:c:\Users\efralmeida\OneDrive - Sefaz-SP\Documentos\Eduardo\Pessoal\Cursos\FIAP ML\Tech Challenge 02\tc2-e-commerce\notebook\..\mlruns
Experiment: baseline_experiments


## Etapa 2 - Carga e preparo dos dados

Usamos apenas interacoes implicitas (view, addtocart, transaction) com peso por tipo de evento.

In [4]:
DATA_PATH = Path("../data/raw/retailrocket/events.csv")
assert DATA_PATH.exists(), f"Arquivo nao encontrado: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df = df[df["event"].isin(EVENT_WEIGHT.keys())].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
df["weight"] = df["event"].map(EVENT_WEIGHT).astype(float)

df = df.sort_values("timestamp").reset_index(drop=True)
df[["visitorid", "itemid"]] = df[["visitorid", "itemid"]].astype("int64")

print(df.head())
print("Interacoes:", len(df))
print("Usuarios:", df["visitorid"].nunique())
print("Itens:", df["itemid"].nunique())

                         timestamp  visitorid      event  itemid  \
0 2015-05-03 03:00:04.384000+00:00     693516  addtocart  297662   
1 2015-05-03 03:00:11.289000+00:00     829044       view   60987   
2 2015-05-03 03:00:13.048000+00:00     652699       view  252860   
3 2015-05-03 03:00:24.154000+00:00    1125936       view   33661   
4 2015-05-03 03:00:26.228000+00:00     693516       view  297662   

   transactionid  weight  
0            NaN     3.0  
1            NaN     1.0  
2            NaN     1.0  
3            NaN     1.0  
4            NaN     1.0  
Interacoes: 2756101
Usuarios: 1407580
Itens: 235061


## Etapa 3 - Split temporal (treino, validacao, teste)

Split temporal evita data leakage em recomendacao.

In [5]:
n = len(df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

print("Train:", train_df["timestamp"].min(), "->", train_df["timestamp"].max(), "|", len(train_df))
print("Valid:", valid_df["timestamp"].min(), "->", valid_df["timestamp"].max(), "|", len(valid_df))
print("Test:", test_df["timestamp"].min(), "->", test_df["timestamp"].max(), "|", len(test_df))

Train: 2015-05-03 03:00:04.384000+00:00 -> 2015-08-02 23:30:53.300000+00:00 | 1929270
Valid: 2015-08-02 23:30:55.147000+00:00 -> 2015-08-25 22:30:59.619000+00:00 | 413415
Test: 2015-08-25 22:31:01.629000+00:00 -> 2015-09-18 02:59:47.788000+00:00 | 413416


## Etapa 4 - Funcoes de avaliacao

Metricas: Recall@K, HitRate@K, NDCG@K e MAP@K.

In [9]:
def build_user_items(frame: pd.DataFrame) -> dict:
    grouped = frame.groupby("visitorid")["itemid"].apply(set)
    return grouped.to_dict()


def sanitize_metric_name(name: str) -> str:
    """MLflow nao aceita '@' em nomes de metricas."""
    return name.replace("@", "_at_")


def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    hits = len(set(rec_k) & relevant)
    return hits / len(relevant)


def hitrate_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    rec_k = set(recommended[:k])
    return 1.0 if len(rec_k & relevant) > 0 else 0.0


def ndcg_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / math.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg


def map_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            hits += 1
            precision_sum += hits / i
    return precision_sum / min(len(relevant), k)


def evaluate_model(user_recs: dict[int, list[int]], truth: dict[int, set[int]], k: int = 10) -> dict[str, float]:
    users = [u for u in truth.keys() if u in user_recs]
    if not users:
        return {"recall@k": 0.0, "hitrate@k": 0.0, "ndcg@k": 0.0, "map@k": 0.0}

    recalls, hits, ndcgs, maps = [], [], [], []
    for user in users:
        recs = user_recs[user]
        rel = truth[user]
        recalls.append(recall_at_k(recs, rel, k))
        hits.append(hitrate_at_k(recs, rel, k))
        ndcgs.append(ndcg_at_k(recs, rel, k))
        maps.append(map_at_k(recs, rel, k))

    return {
        "recall@k": float(np.mean(recalls)),
        "hitrate@k": float(np.mean(hits)),
        "ndcg@k": float(np.mean(ndcgs)),
        "map@k": float(np.mean(maps)),
    }

## Etapa 5 - Modelo dummy (Most Popular)

In [10]:
import time

train_user_items = build_user_items(train_df)
valid_user_items = build_user_items(valid_df)

popular_items = (
    train_df.groupby("itemid")["weight"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

# Otimizacao: reduzir candidatos e avaliar apenas usuarios presentes na validacao
MAX_CANDIDATES = 2000
popular_candidates = popular_items[:MAX_CANDIDATES]
users_to_score = [u for u in valid_user_items.keys() if u in train_user_items]

# Iniciar run do MLflow para Dummy Baseline
with mlflow.start_run(run_name="dummy_most_popular"):
    t0 = time.time()

    # Registrar parâmetros
    mlflow.log_param("model_type", "dummy_most_popular")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("max_candidates", MAX_CANDIDATES)
    mlflow.log_param("users_scored", len(users_to_score))
    mlflow.log_param("train_size", len(train_df))
    mlflow.log_param("valid_size", len(valid_df))
    mlflow.log_param("event_weights", str(EVENT_WEIGHT))

    # Executar modelo
    dummy_recs = {}
    for user in users_to_score:
        seen_items = train_user_items[user]
        recs = [item for item in popular_candidates if item not in seen_items][:TOP_K]
        dummy_recs[user] = recs

    # Avaliar e registrar métricas
    dummy_metrics = evaluate_model(dummy_recs, valid_user_items, k=TOP_K)
    for metric_name, metric_value in dummy_metrics.items():
        mlflow.log_metric(sanitize_metric_name(metric_name), metric_value)

    elapsed = time.time() - t0
    mlflow.log_metric("runtime_seconds", elapsed)

    print("Dummy metrics:", dummy_metrics)
    print(f"Tempo Dummy: {elapsed:.1f}s")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

Dummy metrics: {'recall@k': 0.002816985874845863, 'hitrate@k': 0.0051676861421641, 'ndcg@k': 0.002123801459591312, 'map@k': 0.001522781887486064}
Tempo Dummy: 1.9s
Run ID: 401b51bc29b74c11bc243ec72dc03e0c


## Etapa 6 - Baseline item-item (co-ocorrencia com similaridade cosseno)

Limitacao conhecida: nao funciona para usuarios novos (cold start).

Baseline colaborativo que calcula similaridade entre itens com base no padrao de consumo dos usuarios.Ideia: se dois itens foram consumidos por usuarios parecidos, eles sao similares entre si.

In [12]:
from collections import defaultdict
import time

# Versao escalavel: item-item por co-visitação (sem matriz densa usuario-item)
# Isso evita MemoryError em datasets grandes.

# 1) Reduzir espaço de itens para manter custo sob controle
MAX_ITEMS_BY_POPULARITY = 5000
MAX_ITEMS_PER_USER = 30
TOP_NEIGHBORS = 100

popular_items_for_item_item = (
    train_df.groupby("itemid")["weight"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()[:MAX_ITEMS_BY_POPULARITY]
)
popular_set = set(popular_items_for_item_item)

# 2) Filtrar treino para itens populares e construir histórico por usuário
train_filtered = train_df[train_df["itemid"].isin(popular_set)].copy()
train_user_seq = train_filtered.groupby("visitorid")["itemid"].apply(list).to_dict()

# 3) Co-visitação: conta pares de itens vistos pelo mesmo usuário
co_counts = defaultdict(lambda: defaultdict(float))
for _, items in train_user_seq.items():
    # Remove duplicados preservando ordem e limita tamanho por usuário
    uniq_items = list(dict.fromkeys(items))[:MAX_ITEMS_PER_USER]
    n_items = len(uniq_items)
    for i in range(n_items):
        item_i = uniq_items[i]
        for j in range(i + 1, n_items):
            item_j = uniq_items[j]
            co_counts[item_i][item_j] += 1.0
            co_counts[item_j][item_i] += 1.0

# 4) Mantem apenas vizinhos mais relevantes por item
neighbors = {}
for item_i, related in co_counts.items():
    top_related = sorted(related.items(), key=lambda x: x[1], reverse=True)[:TOP_NEIGHBORS]
    neighbors[item_i] = top_related


def recommend_item_knn_for_user(user: int, top_k: int = 10) -> list[int]:
    """Recomenda por co-visitação item-item (escalável para grande volume)."""
    if user not in train_user_items:
        return []

    seen_items = train_user_items[user]
    scores = defaultdict(float)

    for seen in seen_items:
        for candidate, score in neighbors.get(seen, []):
            if candidate not in seen_items:
                scores[candidate] += score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in ranked[:top_k]]


# Iniciar run do MLflow para Item-Item Baseline (co-visitação)
with mlflow.start_run(run_name="baseline_item_item_covisit"):
    t0 = time.time()

    # Registrar parâmetros
    mlflow.log_param("model_type", "item_item_covisit")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("max_items_by_popularity", MAX_ITEMS_BY_POPULARITY)
    mlflow.log_param("max_items_per_user", MAX_ITEMS_PER_USER)
    mlflow.log_param("top_neighbors", TOP_NEIGHBORS)
    mlflow.log_param("train_size", len(train_df))
    mlflow.log_param("valid_size", len(valid_df))
    mlflow.log_param("event_weights", str(EVENT_WEIGHT))

    # Executar modelo (apenas usuários no ground truth de validação)
    users_to_score_item_item = [u for u in valid_user_items.keys() if u in train_user_items]
    baseline_recs = {
        user: recommend_item_knn_for_user(user, top_k=TOP_K)
        for user in users_to_score_item_item
    }

    # Avaliar e registrar métricas
    baseline_metrics = evaluate_model(baseline_recs, valid_user_items, k=TOP_K)
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(sanitize_metric_name(metric_name), metric_value)

    elapsed = time.time() - t0
    mlflow.log_metric("runtime_seconds", elapsed)

    # Armazenar recomendações como artefato (amostra)
    import json

    recs_sample = {str(u): baseline_recs[u][:5] for u in list(baseline_recs.keys())[:10]}
    with open("baseline_recs_sample.json", "w") as f:
        json.dump(recs_sample, f, indent=2)
    mlflow.log_artifact("baseline_recs_sample.json")

    print("Item-item baseline metrics:", baseline_metrics)
    print(f"Tempo Item-Item: {elapsed:.1f}s")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

Item-item baseline metrics: {'recall@k': 0.015021343495295545, 'hitrate@k': 0.027420375448217675, 'ndcg@k': 0.010773657869910936, 'map@k': 0.007451665465161902}
Tempo Item-Item: 1.3s
Run ID: e25930f5959843b0baf451bd9a78617f


## Etapa 7 - Comparacao dos resultados

In [13]:
# Comparar resultados dos runs via MLflow
experiment = mlflow.get_experiment_by_name("baseline_experiments")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Selecionar apenas colunas relevantes e ordenar por NDCG
comparison_df = runs_df[
    [
        "tags.mlflow.runName",
        "params.model_type",
        "metrics.recall_at_k",
        "metrics.hitrate_at_k",
        "metrics.ndcg_at_k",
        "metrics.map_at_k",
        "metrics.runtime_seconds",
    ]
].sort_values("metrics.ndcg_at_k", ascending=False)

print("\n=== Comparacao de Baselines ===")
print(comparison_df.to_string(index=False))

# ===== ARTEFATOS PARA DVC (Reprodutibilidade) =====
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

# Salvar datasets de treino/validacao/teste (para reprodutibilidade)
train_df.to_csv(processed_dir / "train.csv", index=False)
valid_df.to_csv(processed_dir / "valid.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

# Salvar recomendações dos baselines (artefatos de modelo)
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

import json

with open(models_dir / "dummy_recommendations.json", "w") as f:
    json.dump({str(u): v for u, v in dummy_recs.items()}, f)

with open(models_dir / "baseline_item_item_recommendations.json", "w") as f:
    json.dump({str(u): v for u, v in baseline_recs.items()}, f)

# Salvar métricas detalhadas por modelo
metrics_summary = {
    "dummy_metrics": dummy_metrics,
    "baseline_metrics": baseline_metrics,
    "comparison": comparison_df.to_dict(orient="records"),
}
with open(models_dir / "baseline_metrics_detailed.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

# Salvar CSV para facilitar visualizacao
comparison_df.to_csv(models_dir / "baseline_experiments_comparison.csv", index=False)

print("\nArtefatos salvos para versionamento DVC:")
print(f"   - {processed_dir / 'train.csv'}")
print(f"   - {processed_dir / 'valid.csv'}")
print(f"   - {processed_dir / 'test.csv'}")
print(f"   - {models_dir / 'dummy_recommendations.json'}")
print(f"   - {models_dir / 'baseline_item_item_recommendations.json'}")
print(f"   - {models_dir / 'baseline_metrics_detailed.json'}")
print(f"   - {models_dir / 'baseline_experiments_comparison.csv'}")


=== Comparacao de Baselines ===
       tags.mlflow.runName  params.model_type  metrics.recall_at_k  metrics.hitrate_at_k  metrics.ndcg_at_k  metrics.map_at_k  metrics.runtime_seconds
baseline_item_item_covisit  item_item_covisit             0.015021              0.027420           0.010774          0.007452                 1.257659
        dummy_most_popular dummy_most_popular             0.002817              0.005168           0.002124          0.001523                 1.919553
        dummy_most_popular dummy_most_popular                  NaN                   NaN                NaN               NaN                      NaN
        dummy_most_popular dummy_most_popular                  NaN                   NaN                NaN               NaN                      NaN

Artefatos salvos para versionamento DVC:
   - ..\data\processed\train.csv
   - ..\data\processed\valid.csv
   - ..\data\processed\test.csv
   - ..\models\dummy_recommendations.json
   - ..\models\baseline_item_i

## Etapa 8 - Ajuste de hiperparametros (validacao) e teste final

Nesta etapa:
1. Rodamos uma mini busca de hiperparametros no item-item usando somente validacao.
2. Selecionamos o melhor conjunto por NDCG@K na validacao.
3. Avaliamos no conjunto de teste uma unica vez (dummy e item-item com melhores parametros).

In [14]:
from itertools import product


def run_dummy_for_split(train_hist: dict, eval_truth: dict, top_k: int, max_candidates: int = 2000) -> tuple[dict, dict]:
    """Executa dummy de popularidade e retorna (recs, metrics)."""
    popular = (
        train_df.groupby("itemid")["weight"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()[:max_candidates]
    )

    users_to_score_local = [u for u in eval_truth.keys() if u in train_hist]
    recs_local = {}
    for u in users_to_score_local:
        seen = train_hist[u]
        recs_local[u] = [it for it in popular if it not in seen][:top_k]

    metrics_local = evaluate_model(recs_local, eval_truth, k=top_k)
    return recs_local, metrics_local


def build_item_item_neighbors(
    train_frame: pd.DataFrame,
    max_items_by_popularity: int,
    max_items_per_user: int,
    top_neighbors: int,
) -> dict:
    """Constroi vizinhos por co-visitação para item-item escalável."""
    popular = (
        train_frame.groupby("itemid")["weight"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()[:max_items_by_popularity]
    )
    popular_set_local = set(popular)

    filtered = train_frame[train_frame["itemid"].isin(popular_set_local)].copy()
    user_seq = filtered.groupby("visitorid")["itemid"].apply(list).to_dict()

    co_local = defaultdict(lambda: defaultdict(float))
    for _, items in user_seq.items():
        uniq_items = list(dict.fromkeys(items))[:max_items_per_user]
        n_items_local = len(uniq_items)
        for i in range(n_items_local):
            item_i = uniq_items[i]
            for j in range(i + 1, n_items_local):
                item_j = uniq_items[j]
                co_local[item_i][item_j] += 1.0
                co_local[item_j][item_i] += 1.0

    neighbors_local = {}
    for item_i, related in co_local.items():
        neighbors_local[item_i] = sorted(related.items(), key=lambda x: x[1], reverse=True)[:top_neighbors]

    return neighbors_local


def recommend_item_item_from_neighbors(user: int, neighbors_map: dict, train_hist: dict, top_k: int) -> list[int]:
    if user not in train_hist:
        return []

    seen = train_hist[user]
    scores_local = defaultdict(float)
    for s in seen:
        for candidate, score in neighbors_map.get(s, []):
            if candidate not in seen:
                scores_local[candidate] += score

    ranked = sorted(scores_local.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in ranked[:top_k]]


def run_item_item_for_split(
    train_hist: dict,
    eval_truth: dict,
    top_k: int,
    max_items_by_popularity: int,
    max_items_per_user: int,
    top_neighbors: int,
) -> tuple[dict, dict]:
    neighbors_map = build_item_item_neighbors(
        train_df,
        max_items_by_popularity=max_items_by_popularity,
        max_items_per_user=max_items_per_user,
        top_neighbors=top_neighbors,
    )

    users_to_score_local = [u for u in eval_truth.keys() if u in train_hist]
    recs_local = {
        u: recommend_item_item_from_neighbors(u, neighbors_map, train_hist, top_k)
        for u in users_to_score_local
    }

    metrics_local = evaluate_model(recs_local, eval_truth, k=top_k)
    return recs_local, metrics_local

In [15]:
# Busca de hiperparametros (somente validacao)
search_space = {
    "max_items_by_popularity": [3000, 5000, 8000],
    "max_items_per_user": [20, 30, 50],
    "top_neighbors": [50, 100, 200],
}

grid_results = []
for mip, miu, tn in product(
    search_space["max_items_by_popularity"],
    search_space["max_items_per_user"],
    search_space["top_neighbors"],
):
    run_name = f"hparam_item_item_mip{mip}_miu{miu}_tn{tn}"
    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        recs_valid, metrics_valid = run_item_item_for_split(
            train_hist=train_user_items,
            eval_truth=valid_user_items,
            top_k=TOP_K,
            max_items_by_popularity=mip,
            max_items_per_user=miu,
            top_neighbors=tn,
        )

        mlflow.log_param("model_type", "item_item_covisit")
        mlflow.log_param("top_k", TOP_K)
        mlflow.log_param("max_items_by_popularity", mip)
        mlflow.log_param("max_items_per_user", miu)
        mlflow.log_param("top_neighbors", tn)
        mlflow.log_param("eval_split", "validation")

        for metric_name, metric_value in metrics_valid.items():
            mlflow.log_metric(sanitize_metric_name(metric_name), metric_value)

        elapsed = time.time() - t0
        mlflow.log_metric("runtime_seconds", elapsed)

        grid_results.append(
            {
                "run_name": run_name,
                "max_items_by_popularity": mip,
                "max_items_per_user": miu,
                "top_neighbors": tn,
                "recall@k": metrics_valid["recall@k"],
                "hitrate@k": metrics_valid["hitrate@k"],
                "ndcg@k": metrics_valid["ndcg@k"],
                "map@k": metrics_valid["map@k"],
                "runtime_seconds": elapsed,
            }
        )

grid_results_df = pd.DataFrame(grid_results).sort_values("ndcg@k", ascending=False).reset_index(drop=True)
display(grid_results_df.head(10))

best_cfg = grid_results_df.iloc[0].to_dict()
print("\nMelhor configuracao na validacao:")
print(best_cfg)

,run_name,max_items_by_popularity,max_items_per_user,top_neighbors,recall@k,hitrate@k,ndcg@k,map@k,runtime_seconds
0,hparam_item_item_mip8000_miu30_tn50,8000,30,50,0.019513,0.034328,0.013668,0.009459,10.907522
1,hparam_item_item_mip8000_miu30_tn200,8000,30,200,0.019432,0.034275,0.013651,0.009463,13.364732
2,hparam_item_item_mip8000_miu30_tn100,8000,30,100,0.019465,0.034117,0.013647,0.009458,13.101269
3,hparam_item_item_mip8000_miu20_tn200,8000,20,200,0.019275,0.034012,0.013595,0.009456,13.374204
4,hparam_item_item_mip8000_miu20_tn100,8000,20,100,0.019284,0.034012,0.013594,0.009452,14.216173
5,hparam_item_item_mip8000_miu20_tn50,8000,20,50,0.019273,0.033748,0.013577,0.009448,10.999556
6,hparam_item_item_mip8000_miu50_tn50,8000,50,50,0.019265,0.034065,0.013573,0.009406,32.698744
7,hparam_item_item_mip8000_miu50_tn200,8000,50,200,0.019252,0.034012,0.013539,0.009375,25.864468
8,hparam_item_item_mip8000_miu50_tn100,8000,50,100,0.019226,0.034012,0.013537,0.009376,21.140694
9,hparam_item_item_mip5000_miu50_tn200,5000,50,200,0.015191,0.027737,0.010870,0.007501,8.794212



Melhor configuracao na validacao:
{'run_name': 'hparam_item_item_mip8000_miu30_tn50', 'max_items_by_popularity': 8000, 'max_items_per_user': 30, 'top_neighbors': 50, 'recall@k': 0.019513221596268823, 'hitrate@k': 0.03432820080151867, 'ndcg@k': 0.01366818574678765, 'map@k': 0.009458996248491194, 'runtime_seconds': 10.907522201538086}


In [16]:
# Teste final (uma unica vez) com a melhor configuracao da validacao
best_mip = int(best_cfg["max_items_by_popularity"])
best_miu = int(best_cfg["max_items_per_user"])
best_tn = int(best_cfg["top_neighbors"])

# 1) Dummy no teste
with mlflow.start_run(run_name="final_test_dummy"):
    t0 = time.time()
    test_dummy_recs, test_dummy_metrics = run_dummy_for_split(
        train_hist=train_user_items,
        eval_truth=build_user_items(test_df),
        top_k=TOP_K,
        max_candidates=MAX_CANDIDATES,
    )

    mlflow.log_param("model_type", "dummy_most_popular")
    mlflow.log_param("eval_split", "test")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("max_candidates", MAX_CANDIDATES)

    for metric_name, metric_value in test_dummy_metrics.items():
        mlflow.log_metric(sanitize_metric_name(metric_name), metric_value)

    elapsed = time.time() - t0
    mlflow.log_metric("runtime_seconds", elapsed)

# 2) Item-item no teste com melhor hiperparametro
with mlflow.start_run(run_name="final_test_item_item_best"):
    t0 = time.time()
    test_item_recs, test_item_metrics = run_item_item_for_split(
        train_hist=train_user_items,
        eval_truth=build_user_items(test_df),
        top_k=TOP_K,
        max_items_by_popularity=best_mip,
        max_items_per_user=best_miu,
        top_neighbors=best_tn,
    )

    mlflow.log_param("model_type", "item_item_covisit")
    mlflow.log_param("eval_split", "test")
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("max_items_by_popularity", best_mip)
    mlflow.log_param("max_items_per_user", best_miu)
    mlflow.log_param("top_neighbors", best_tn)

    for metric_name, metric_value in test_item_metrics.items():
        mlflow.log_metric(sanitize_metric_name(metric_name), metric_value)

    elapsed = time.time() - t0
    mlflow.log_metric("runtime_seconds", elapsed)

# Quadro final de teste
final_test_df = pd.DataFrame(
    [
        {"model": "dummy_test", **test_dummy_metrics},
        {"model": "item_item_test_best", **test_item_metrics},
    ]
).sort_values("ndcg@k", ascending=False)

display(final_test_df)

print("\nResumo teste final:")
print(final_test_df.to_string(index=False))

,model,recall@k,hitrate@k,ndcg@k,map@k
1,item_item_test_best,0.01037,0.019724,0.007102,0.004616
0,dummy_test,0.00211,0.003540,0.001684,0.001330



Resumo teste final:
              model  recall@k  hitrate@k   ndcg@k    map@k
item_item_test_best   0.01037   0.019724 0.007102 0.004616
         dummy_test   0.00211   0.003540 0.001684 0.001330
